# Bloque 04 — Ajuste de cópulas para eventos de sequía nSPEI-3

Versión corregida para mantener la trazabilidad con el ajuste marginal definitivo.

Flujo:

1. Lee los eventos nSPEI-3.
2. Lee las distribuciones marginales seleccionadas en el Bloque 03.
3. Usa los parámetros MLE finales del Bloque 03 para transformar duración y severidad al espacio uniforme.
4. Ajusta cinco familias de cópulas: Clayton, Frank, Gumbel, Gaussiana y t-Student.
5. Selecciona por AIC y contrasta con BIC.
6. Calcula medidas de dependencia y dependencia de colas de la cópula seleccionada.
7. Genera tablas y figuras de forma dinámica: no se presupone que Gumbel sea la mejor familia.

Importante: las observaciones transformadas se obtienen mediante las CDF marginales paramétricas ajustadas, no mediante rangos empíricos.


In [ ]:
# ============================================================
# 1. LIBRERÍAS
# ============================================================

if (!require("pacman")) install.packages("pacman")

pacman::p_load(
  tidyverse,
  janitor,
  lmomco,
  copula,
  patchwork,
  plotly,
  htmlwidgets,
  viridisLite,
  cowplot
)


In [ ]:
# ============================================================
# 2. RUTAS DEL PROYECTO
# ============================================================

encontrar_raiz_proyecto <- function(inicio = getwd()) {
  marcadores <- c("01_datos", "02_scripts", "03_resultados", "04_documentacion")
  es_raiz <- function(x) all(dir.exists(file.path(x, marcadores)))

  raiz_env <- Sys.getenv("TESIS_SEQUIAS_ROOT", unset = "")
  if (nzchar(raiz_env)) {
    raiz_env <- normalizePath(raiz_env, winslash = "/", mustWork = FALSE)
    if (es_raiz(raiz_env)) return(raiz_env)
  }

  ruta <- normalizePath(inicio, winslash = "/", mustWork = TRUE)

  repeat {
    if (es_raiz(ruta)) return(ruta)

    padre <- dirname(ruta)
    if (identical(padre, ruta)) {
      stop(
        "No se encontró la raíz del proyecto. ",
        "Ejecute Jupyter dentro del repositorio o defina TESIS_SEQUIAS_ROOT."
      )
    }

    ruta <- padre
  }
}

ruta_base <- encontrar_raiz_proyecto()

ruta_procesados <- file.path(
  ruta_base, "01_datos", "procesados"
)

ruta_marginales <- file.path(
  ruta_base, "03_resultados", "03_marginales", "tablas"
)

ruta_tablas <- file.path(
  ruta_base, "03_resultados", "04_copulas", "tablas"
)

ruta_figuras <- file.path(
  ruta_base, "03_resultados", "04_copulas", "figuras"
)

archivo_eventos <- file.path(
  ruta_procesados,
  "eventos_nspei_todas_escalas.csv"
)

archivo_marginales <- file.path(
  ruta_marginales,
  "mejores_marginales_nspei3_por_cluster.csv"
)

dir.create(ruta_tablas, recursive = TRUE, showWarnings = FALSE)
dir.create(ruta_figuras, recursive = TRUE, showWarnings = FALSE)

cat("Archivo de eventos:", file.exists(archivo_eventos), "\n")
cat("Marginales finales:", file.exists(archivo_marginales), "\n")
cat("Ruta tablas:", ruta_tablas, "\n")
cat("Ruta figuras:", ruta_figuras, "\n")

if (!file.exists(archivo_eventos)) {
  stop("No se encontró el archivo de eventos.")
}

if (!file.exists(archivo_marginales)) {
  stop(
    "No se encontró el archivo de marginales finales. ",
    "Ejecute primero el Bloque 03 MLE corregido."
  )
}


In [ ]:
# ============================================================
# 3. EVENTOS DE SEQUÍA nSPEI-3
# ============================================================

eventos <- read.csv(
  archivo_eventos,
  stringsAsFactors = FALSE
) %>%
  clean_names() %>%
  mutate(
    fecha_inicio = as.Date(fecha_inicio),
    fecha_fin = as.Date(fecha_fin),
    cluster_ward = as.integer(cluster_ward)
  )

eventos_3 <- eventos %>%
  filter(escala == "nspei_3") %>%
  filter(
    is.finite(duracion),
    is.finite(severidad),
    duracion > 0,
    severidad > 0
  )

tabla_eventos_copulas <- eventos_3 %>%
  count(cluster_ward, name = "numero_eventos") %>%
  transmute(
    cluster = cluster_ward,
    numero_eventos,
    escala = "nSPEI-3"
  )

write.csv(
  tabla_eventos_copulas,
  file.path(ruta_tablas, "eventos_ajuste_bivariado_nspei3.csv"),
  row.names = FALSE
)

tabla_eventos_copulas


In [ ]:
# ============================================================
# 4. MARGINALES MLE FINALES Y TRANSFORMACIÓN AL ESPACIO UNIFORME
# ============================================================

marginales_finales <- read.csv(
  archivo_marginales,
  stringsAsFactors = FALSE,
  check.names = FALSE
) %>%
  clean_names()

# Se esperan las selecciones definitivas obtenidas en el Bloque 03:
# C1: D=LN3, S=LN3
# C2: D=LN3, S=LN3
# C3: D=LN3, S=GPA
esperado <- tibble(
  cluster = c(1, 1, 2, 2, 3, 3),
  variable_clave = c("D", "S", "D", "S", "D", "S"),
  modelo_esperado = c("LN3", "LN3", "LN3", "LN3", "LN3", "GPA")
)

# Normalización de columnas del CSV del Bloque 03.
marginales_finales <- marginales_finales %>%
  mutate(
    cluster = as.integer(stringr::str_extract(grupo, "\\d+")),
    variable_clave = if_else(
      stringr::str_detect(variable, regex("Duraci", ignore_case = TRUE)),
      "D",
      "S"
    ),
    modelo = toupper(modelo)
  )

verificacion_modelos <- marginales_finales %>%
  select(cluster, variable_clave, modelo) %>%
  left_join(
    esperado,
    by = c("cluster", "variable_clave")
  ) %>%
  mutate(coincide = modelo == modelo_esperado)

print(verificacion_modelos)

if (any(
  is.na(verificacion_modelos$coincide) |
  !verificacion_modelos$coincide
)) {
  stop(
    "Las marginales encontradas no coinciden con las marginales definitivas ",
    "del Bloque 03. Revise antes de continuar."
  )
}

# Convierte la cadena "p1, p2, p3" a vector numérico.
parsear_parametros <- function(x) {
  as.numeric(trimws(strsplit(x, ",", fixed = TRUE)[[1]]))
}

obtener_marginal <- function(cluster, variable_clave) {

  fila <- marginales_finales %>%
    filter(
      .data$cluster == .env$cluster,
      .data$variable_clave == .env$variable_clave
    )

  if (nrow(fila) != 1) {
    stop(
      "No se encontró exactamente una marginal para cluster ",
      cluster, " y variable ", variable_clave
    )
  }

  tipo <- tolower(fila$modelo[[1]])
  pars <- parsear_parametros(fila$parametros[[1]])

  par_obj <- lmomco::vec2par(
    pars,
    type = tipo
  )

  list(
    modelo = toupper(tipo),
    parametros = pars,
    par_obj = par_obj
  )
}

transformar_cluster <- function(datos, cluster, eps = 1e-8) {

  # Mantener duración y severidad como pares del mismo evento.
  datos_validos <- datos %>%
    transmute(
      duracion = as.numeric(duracion),
      severidad = as.numeric(severidad)
    ) %>%
    filter(
      is.finite(duracion),
      is.finite(severidad),
      duracion > 0,
      severidad > 0
    )

  md <- obtener_marginal(cluster, "D")
  ms <- obtener_marginal(cluster, "S")

  u_raw <- lmomco::plmomco(
    datos_validos$duracion,
    md$par_obj
  )

  v_raw <- lmomco::plmomco(
    datos_validos$severidad,
    ms$par_obj
  )

  if (
    any(!is.finite(u_raw)) ||
    any(!is.finite(v_raw))
  ) {
    stop(
      "Se obtuvieron CDF no finitas en el cluster ",
      cluster
    )
  }

  n_clip_u <- sum(u_raw <= eps | u_raw >= 1 - eps)
  n_clip_v <- sum(v_raw <= eps | v_raw >= 1 - eps)

  u <- pmin(pmax(u_raw, eps), 1 - eps)
  v <- pmin(pmax(v_raw, eps), 1 - eps)

  list(
    datos = tibble(
      duracion = datos_validos$duracion,
      severidad = datos_validos$severidad,
      u = u,
      v = v
    ),
    marginal_d = md$modelo,
    marginal_s = ms$modelo,
    n_clip_u = n_clip_u,
    n_clip_v = n_clip_v
  )
}

datos_c1 <- eventos_3 %>% filter(cluster_ward == 1)
datos_c2 <- eventos_3 %>% filter(cluster_ward == 2)
datos_c3 <- eventos_3 %>% filter(cluster_ward == 3)

tr_c1 <- transformar_cluster(datos_c1, 1)
tr_c2 <- transformar_cluster(datos_c2, 2)
tr_c3 <- transformar_cluster(datos_c3, 3)

pseudo_c1 <- tr_c1$datos
pseudo_c2 <- tr_c2$datos
pseudo_c3 <- tr_c3$datos

diagnostico_transformacion <- tibble(
  cluster = 1:3,
  marginal_duracion = c(tr_c1$marginal_d, tr_c2$marginal_d, tr_c3$marginal_d),
  marginal_severidad = c(tr_c1$marginal_s, tr_c2$marginal_s, tr_c3$marginal_s),
  n = c(nrow(pseudo_c1), nrow(pseudo_c2), nrow(pseudo_c3)),
  n_recortes_u = c(tr_c1$n_clip_u, tr_c2$n_clip_u, tr_c3$n_clip_u),
  n_recortes_v = c(tr_c1$n_clip_v, tr_c2$n_clip_v, tr_c3$n_clip_v),
  min_u = c(min(pseudo_c1$u), min(pseudo_c2$u), min(pseudo_c3$u)),
  max_u = c(max(pseudo_c1$u), max(pseudo_c2$u), max(pseudo_c3$u)),
  min_v = c(min(pseudo_c1$v), min(pseudo_c2$v), min(pseudo_c3$v)),
  max_v = c(max(pseudo_c1$v), max(pseudo_c2$v), max(pseudo_c3$v))
)

write.csv(
  diagnostico_transformacion,
  file.path(ruta_tablas, "diagnostico_transformacion_uniforme_nspei3.csv"),
  row.names = FALSE
)

diagnostico_transformacion


In [ ]:
# ============================================================
# 5. AJUSTE DE CINCO FAMILIAS DE CÓPULAS
# ============================================================

ajustar_una_copula <- function(familia, uv) {

  tau_emp <- cor(
    uv[, 1],
    uv[, 2],
    method = "kendall",
    use = "complete.obs"
  )

  rho_ini <- sin(pi * tau_emp / 2)
  rho_ini <- min(max(rho_ini, -0.98), 0.98)

  crear <- switch(
    familia,

    "Clayton" = function() {
      theta0 <- tryCatch(
        iTau(claytonCopula(dim = 2), tau_emp),
        error = function(e) 1
      )
      list(
        cop = claytonCopula(dim = 2),
        start = max(theta0, 1e-4)
      )
    },

    "Frank" = function() {
      theta0 <- tryCatch(
        iTau(frankCopula(dim = 2), tau_emp),
        error = function(e) 1
      )
      list(
        cop = frankCopula(dim = 2),
        start = theta0
      )
    },

    "Gumbel" = function() {
      theta0 <- tryCatch(
        iTau(gumbelCopula(dim = 2), max(tau_emp, 1e-5)),
        error = function(e) 1.1
      )
      list(
        cop = gumbelCopula(dim = 2),
        start = max(theta0, 1.0001)
      )
    },

    "Gaussian" = function() {
      list(
        cop = normalCopula(dim = 2, dispstr = "un"),
        start = rho_ini
      )
    },

    "t-Student" = function() {
      list(
        cop = tCopula(
          dim = 2,
          dispstr = "un",
          df = 6,
          df.fixed = FALSE
        ),
        start = c(rho_ini, 6)
      )
    },

    stop("Familia no implementada: ", familia)
  )

  spec <- crear()

  fit <- tryCatch(
    suppressWarnings(
      fitCopula(
        spec$cop,
        uv,
        method = "ml",
        start = spec$start,
        estimate.variance = TRUE
      )
    ),
    error = function(e) NULL
  )

  if (is.null(fit)) {
    return(
      list(
        fila = tibble(
          familia = familia,
          logLik = NA_real_,
          AIC = Inf,
          BIC = Inf,
          k = NA_integer_,
          convergio = FALSE,
          parametro = NA_character_
        ),
        fit = NULL
      )
    )
  }

  ll <- as.numeric(logLik(fit))
  pars <- coef(fit)
  k <- length(pars)

  codigo <- tryCatch(
    fit@fitting.stats$convergence,
    error = function(e) 0
  )

  convergio <- is.finite(ll) && (is.null(codigo) || codigo == 0)

  fila <- tibble(
    familia = familia,
    logLik = ll,
    AIC = ifelse(convergio, -2 * ll + 2 * k, Inf),
    BIC = ifelse(convergio, -2 * ll + log(nrow(uv)) * k, Inf),
    k = k,
    convergio = convergio,
    parametro = paste(
      paste0(names(pars), "=", round(as.numeric(pars), 6)),
      collapse = "; "
    )
  )

  list(
    fila = fila,
    fit = if (convergio) fit else NULL
  )
}

ajustar_copulas_candidatas <- function(u, v) {

  uv <- cbind(u, v)

  if (
    any(!is.finite(uv)) ||
    any(uv <= 0) ||
    any(uv >= 1)
  ) {
    stop("Las observaciones transformadas deben pertenecer a (0,1).")
  }

  familias <- c(
    "Clayton",
    "Frank",
    "Gumbel",
    "Gaussian",
    "t-Student"
  )

  ajustes <- lapply(
    familias,
    ajustar_una_copula,
    uv = uv
  )

  tabla <- bind_rows(
    lapply(ajustes, `[[`, "fila")
  ) %>%
    arrange(AIC)

  modelos <- setNames(
    lapply(ajustes, `[[`, "fit"),
    familias
  )

  list(
    tabla = tabla,
    modelos = modelos
  )
}


In [ ]:
# ============================================================
# 6. AJUSTE DE CÓPULAS POR CLUSTER
# ============================================================

cop_c1 <- ajustar_copulas_candidatas(pseudo_c1$u, pseudo_c1$v)
cop_c2 <- ajustar_copulas_candidatas(pseudo_c2$u, pseudo_c2$v)
cop_c3 <- ajustar_copulas_candidatas(pseudo_c3$u, pseudo_c3$v)

cat(
  "Cluster 1:", nrow(pseudo_c1), "eventos |",
  tr_c1$marginal_d, "/", tr_c1$marginal_s, "\n"
)

cat(
  "Cluster 2:", nrow(pseudo_c2), "eventos |",
  tr_c2$marginal_d, "/", tr_c2$marginal_s, "\n"
)

cat(
  "Cluster 3:", nrow(pseudo_c3), "eventos |",
  tr_c3$marginal_d, "/", tr_c3$marginal_s, "\n"
)


In [ ]:
# ============================================================
# 7. COMPARACIÓN DE FAMILIAS DE CÓPULAS
# ============================================================

tabla_comparacion_copulas <- bind_rows(
  cop_c1$tabla %>% mutate(cluster = 1),
  cop_c2$tabla %>% mutate(cluster = 2),
  cop_c3$tabla %>% mutate(cluster = 3)
) %>%
  select(
    cluster,
    familia,
    logLik,
    AIC,
    BIC,
    k,
    convergio,
    parametro
  ) %>%
  arrange(cluster, AIC)

write.csv(
  tabla_comparacion_copulas,
  file.path(ruta_tablas, "comparacion_copulas_nspei3.csv"),
  row.names = FALSE
)

tabla_comparacion_copulas


In [ ]:
# ============================================================
# 8. CÓPULAS SELECCIONADAS Y MEDIDAS DE DEPENDENCIA
# ============================================================

validos <- tabla_comparacion_copulas %>%
  filter(
    convergio,
    is.finite(AIC),
    is.finite(BIC),
    is.finite(logLik)
  )

mejor_aic <- validos %>%
  group_by(cluster) %>%
  slice_min(AIC, n = 1, with_ties = FALSE) %>%
  ungroup()

mejor_bic <- validos %>%
  group_by(cluster) %>%
  slice_min(BIC, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  select(
    cluster,
    familia_bic = familia,
    bic_min = BIC
  )

tabla_copulas_seleccionadas <- mejor_aic %>%
  left_join(
    mejor_bic,
    by = "cluster"
  ) %>%
  mutate(
    coincide_aic_bic = familia == familia_bic
  )

if (any(!tabla_copulas_seleccionadas$coincide_aic_bic)) {
  warning(
    "AIC y BIC no seleccionaron la misma familia en al menos un cluster."
  )
}

obtener_fit_seleccionado <- function(cluster) {

  fam <- tabla_copulas_seleccionadas %>%
    filter(.data$cluster == .env$cluster) %>%
    pull(familia)

  lista <- switch(
    as.character(cluster),
    "1" = cop_c1$modelos,
    "2" = cop_c2$modelos,
    "3" = cop_c3$modelos
  )

  fit <- lista[[fam]]

  if (is.null(fit)) {
    stop("No se encontró el objeto ajustado para el cluster ", cluster)
  }

  fit
}

fits_seleccionados <- lapply(
  1:3,
  obtener_fit_seleccionado
)

medidas_dependencia <- map_dfr(
  1:3,
  function(cl) {

    fit <- fits_seleccionados[[cl]]
    cop <- fit@copula

    lam <- tryCatch(
      copula::lambda(cop),
      error = function(e) c(lower = NA_real_, upper = NA_real_)
    )

    tibble(
      cluster = cl,
      tau_kendall_empirico = cor(
        switch(
          as.character(cl),
          "1" = pseudo_c1$u,
          "2" = pseudo_c2$u,
          "3" = pseudo_c3$u
        ),
        switch(
          as.character(cl),
          "1" = pseudo_c1$v,
          "2" = pseudo_c2$v,
          "3" = pseudo_c3$v
        ),
        method = "kendall"
      ),
      tau_kendall_modelo = tryCatch(
        copula::tau(cop),
        error = function(e) NA_real_
      ),
      lambda_inferior = unname(lam[1]),
      lambda_superior = unname(lam[2])
    )
  }
)

tabla_copulas_seleccionadas <- tabla_copulas_seleccionadas %>%
  left_join(
    medidas_dependencia,
    by = "cluster"
  )

write.csv(
  tabla_copulas_seleccionadas,
  file.path(ruta_tablas, "copulas_seleccionadas_nspei3.csv"),
  row.names = FALSE
)

tabla_copulas_seleccionadas


In [ ]:
# ============================================================
# 9. RESUMEN VISIBLE PARA LA TESIS
# ============================================================

cat("\n=== TRANSFORMACIÓN MARGINAL ===\n")
print(
  diagnostico_transformacion,
  n = Inf,
  width = Inf
)

cat("\n=== COMPARACIÓN DE CÓPULAS ===\n")
print(
  tabla_comparacion_copulas,
  n = Inf,
  width = Inf
)

cat("\n=== CÓPULAS SELECCIONADAS ===\n")
print(
  tabla_copulas_seleccionadas,
  n = Inf,
  width = Inf
)


In [ ]:
# ============================================================
# 10. FIGURA - OBSERVACIONES TRANSFORMADAS AL ESPACIO UNIFORME
# ============================================================

crear_plot_uv <- function(df, cluster_label) {
  ggplot(df, aes(u, v)) +
    geom_point(alpha = 0.45, size = 1.1) +
    coord_equal(
      xlim = c(0, 1),
      ylim = c(0, 1)
    ) +
    theme_minimal(base_size = 11) +
    labs(
      title = cluster_label,
      x = "u = F_D(d)",
      y = "v = F_S(s)"
    )
}

p_uv_c1 <- crear_plot_uv(pseudo_c1, "(a) Cluster 1")
p_uv_c2 <- crear_plot_uv(pseudo_c2, "(b) Cluster 2")
p_uv_c3 <- crear_plot_uv(pseudo_c3, "(c) Cluster 3")

figura_uv <- (p_uv_c1 | p_uv_c2 | p_uv_c3) +
  plot_annotation(
    title = "Observaciones transformadas de duración y severidad por cluster"
  )

ggsave(
  file.path(ruta_figuras, "observaciones_uniformes_nspei3.png"),
  figura_uv,
  width = 13.5,
  height = 4.8,
  dpi = 400,
  bg = "white"
)

ggsave(
  file.path(ruta_figuras, "observaciones_uniformes_cluster1.png"),
  p_uv_c1,
  width = 5.2,
  height = 4.8,
  dpi = 400,
  bg = "white"
)

ggsave(
  file.path(ruta_figuras, "observaciones_uniformes_cluster2.png"),
  p_uv_c2,
  width = 5.2,
  height = 4.8,
  dpi = 400,
  bg = "white"
)

ggsave(
  file.path(ruta_figuras, "observaciones_uniformes_cluster3.png"),
  p_uv_c3,
  width = 5.2,
  height = 4.8,
  dpi = 400,
  bg = "white"
)

figura_uv


In [ ]:
# ============================================================
# 11. SUPERFICIES BIDIMENSIONALES DE LAS CÓPULAS SELECCIONADAS
# ============================================================

crear_superficie_copula <- function(fit, cluster_label, n_grid = 180) {

  grid <- expand.grid(
    u = seq(0.001, 0.999, length.out = n_grid),
    v = seq(0.001, 0.999, length.out = n_grid)
  )

  cop <- fit@copula

  grid$C <- pCopula(
    as.matrix(grid[, c("u", "v")]),
    cop
  )

  as_tibble(grid) %>%
    mutate(cluster = cluster_label)
}

plot_copula_2d <- function(surface_df, pseudo_df, familia, cluster_label) {

  ggplot() +
    geom_raster(
      data = surface_df,
      aes(x = u, y = v, fill = C),
      interpolate = TRUE
    ) +
    geom_contour(
      data = surface_df,
      aes(x = u, y = v, z = C),
      colour = "white",
      bins = 7,
      linewidth = 0.4,
      alpha = 0.9
    ) +
    geom_point(
      data = pseudo_df,
      aes(x = u, y = v),
      size = 0.9,
      alpha = 0.38
    ) +
    coord_equal(expand = FALSE) +
    scale_fill_gradientn(
      colours = rev(viridisLite::magma(256)),
      name = "C(u,v)"
    ) +
    theme_minimal(base_size = 11) +
    theme(
      legend.position = "right",
      panel.grid.minor = element_blank()
    ) +
    labs(
      title = cluster_label,
      subtitle = paste("Cópula seleccionada:", familia),
      x = "u = F_D(d)",
      y = "v = F_S(s)"
    )
}

familias_sel <- tabla_copulas_seleccionadas %>%
  arrange(cluster) %>%
  pull(familia)

surf_c1 <- crear_superficie_copula(
  fits_seleccionados[[1]], "Cluster 1"
)
surf_c2 <- crear_superficie_copula(
  fits_seleccionados[[2]], "Cluster 2"
)
surf_c3 <- crear_superficie_copula(
  fits_seleccionados[[3]], "Cluster 3"
)

p2d_c1 <- plot_copula_2d(
  surf_c1, pseudo_c1, familias_sel[1], "Cluster 1"
)
p2d_c2 <- plot_copula_2d(
  surf_c2, pseudo_c2, familias_sel[2], "Cluster 2"
)
p2d_c3 <- plot_copula_2d(
  surf_c3, pseudo_c3, familias_sel[3], "Cluster 3"
)

figura_2d <- (p2d_c1 | p2d_c2 | p2d_c3) +
  plot_annotation(
    title = "Superficies bidimensionales de las cópulas seleccionadas por cluster"
  )

ggsave(
  file.path(ruta_figuras, "copulas_seleccionadas_2d_nspei3.png"),
  figura_2d,
  width = 15.0,
  height = 5.3,
  dpi = 450,
  bg = "white"
)

ggsave(
  file.path(ruta_figuras, "copula_2d_cluster1.png"),
  p2d_c1,
  width = 5.4,
  height = 5.0,
  dpi = 450,
  bg = "white"
)

ggsave(
  file.path(ruta_figuras, "copula_2d_cluster2.png"),
  p2d_c2,
  width = 5.4,
  height = 5.0,
  dpi = 450,
  bg = "white"
)

ggsave(
  file.path(ruta_figuras, "copula_2d_cluster3.png"),
  p2d_c3,
  width = 5.4,
  height = 5.0,
  dpi = 450,
  bg = "white"
)

figura_2d


In [ ]:
# ============================================================
# 12. SUPERFICIES TRIDIMENSIONALES DE LAS CÓPULAS SELECCIONADAS
# ============================================================

guardar_superficie_3d <- function(
    surface_df,
    pseudo_df,
    fit,
    familia,
    cluster_label,
    archivo_png,
    archivo_html
) {

  u_vals <- sort(unique(surface_df$u))
  v_vals <- sort(unique(surface_df$v))

  z_mat <- matrix(
    surface_df$C,
    nrow = length(u_vals),
    ncol = length(v_vals),
    byrow = FALSE
  )

  cop <- fit@copula

  z_pts <- pCopula(
    as.matrix(pseudo_df[, c("u", "v")]),
    cop
  )

  # ---------- PNG estático para la tesis ----------
  png(
    archivo_png,
    width = 2200,
    height = 1800,
    res = 300
  )

  pm <- persp(
    x = u_vals,
    y = v_vals,
    z = z_mat,
    theta = 38,
    phi = 28,
    expand = 0.75,
    ticktype = "detailed",
    xlab = "u = F_D(d)",
    ylab = "v = F_S(s)",
    zlab = "C(u,v)",
    main = paste0(
      cluster_label,
      " - Cópula ",
      familia
    )
  )

  pts <- trans3d(
    x = pseudo_df$u,
    y = pseudo_df$v,
    z = z_pts,
    pmat = pm
  )

  points(
    pts$x,
    pts$y,
    pch = 16,
    cex = 0.35
  )

  dev.off()

  # ---------- HTML interactivo como respaldo ----------
  fig_html <- plot_ly() %>%
    add_surface(
      x = ~u_vals,
      y = ~v_vals,
      z = ~z_mat,
      opacity = 0.92,
      showscale = TRUE,
      colorbar = list(title = "C(u,v)"),
      name = "Superficie"
    ) %>%
    add_markers(
      x = pseudo_df$u,
      y = pseudo_df$v,
      z = z_pts,
      marker = list(
        size = 2.5,
        color = "black",
        opacity = 0.55
      ),
      name = "Observaciones transformadas"
    ) %>%
    layout(
      title = list(
        text = paste0(
          cluster_label,
          " | Cópula ",
          familia
        ),
        x = 0.5
      ),
      scene = list(
        xaxis = list(title = "u = F_D(d)"),
        yaxis = list(title = "v = F_S(s)"),
        zaxis = list(title = "C(u,v)")
      )
    )

  htmlwidgets::saveWidget(
    fig_html,
    archivo_html,
    selfcontained = FALSE
  )

  invisible(fig_html)
}

guardar_superficie_3d(
  surf_c1,
  pseudo_c1,
  fits_seleccionados[[1]],
  familias_sel[1],
  "Cluster 1",
  file.path(ruta_figuras, "copula_3d_cluster1.png"),
  file.path(ruta_figuras, "copula_3d_cluster1.html")
)

guardar_superficie_3d(
  surf_c2,
  pseudo_c2,
  fits_seleccionados[[2]],
  familias_sel[2],
  "Cluster 2",
  file.path(ruta_figuras, "copula_3d_cluster2.png"),
  file.path(ruta_figuras, "copula_3d_cluster2.html")
)

guardar_superficie_3d(
  surf_c3,
  pseudo_c3,
  fits_seleccionados[[3]],
  familias_sel[3],
  "Cluster 3",
  file.path(ruta_figuras, "copula_3d_cluster3.png"),
  file.path(ruta_figuras, "copula_3d_cluster3.html")
)

cat("Superficies 3D guardadas en PNG y HTML.\n")


In [ ]:
# Resumen de familias utilizadas en las figuras.
tibble(
  cluster = 1:3,
  familia = familias_sel
)


In [ ]:
# Medidas de dependencia de las cópulas seleccionadas.
medidas_dependencia


In [ ]:
# ============================================================
# 13. VERIFICACIÓN FINAL DE SALIDAS
# ============================================================

salidas <- c(
  file.path(ruta_tablas, "eventos_ajuste_bivariado_nspei3.csv"),
  file.path(ruta_tablas, "diagnostico_transformacion_uniforme_nspei3.csv"),
  file.path(ruta_tablas, "comparacion_copulas_nspei3.csv"),
  file.path(ruta_tablas, "copulas_seleccionadas_nspei3.csv"),
  file.path(ruta_figuras, "observaciones_uniformes_nspei3.png"),
  file.path(ruta_figuras, "copulas_seleccionadas_2d_nspei3.png"),
  file.path(ruta_figuras, "copula_2d_cluster1.png"),
  file.path(ruta_figuras, "copula_2d_cluster2.png"),
  file.path(ruta_figuras, "copula_2d_cluster3.png"),
  file.path(ruta_figuras, "copula_3d_cluster1.png"),
  file.path(ruta_figuras, "copula_3d_cluster2.png"),
  file.path(ruta_figuras, "copula_3d_cluster3.png"),
  file.path(ruta_figuras, "copula_3d_cluster1.html"),
  file.path(ruta_figuras, "copula_3d_cluster2.html"),
  file.path(ruta_figuras, "copula_3d_cluster3.html")
)

verificacion <- tibble(
  archivo = basename(salidas),
  existe = file.exists(salidas),
  tamaño_bytes = file.info(salidas)$size
)

cat("\n=== VERIFICACIÓN FINAL ===\n")
print(verificacion, n = Inf, width = Inf)
